In [0]:
# %skip
%pip install databricks-labs-dqx
dbutils.library.restartPython()

In [0]:
# %skip
import databricks.labs.dqx.profiler.profiler as dqx_module
print(dir(dqx_module))
from databricks.labs.dqx.profiler.profiler import DQProfiler
import inspect
print(inspect.signature(DQProfiler.__init__))

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.labs.dqx.profiler.profiler import DQProfiler

w = WorkspaceClient()
profiler = DQProfiler(workspace_client=w, spark=spark)

print("DQX Profiler initialized!")

In [0]:
chicago = spark.table("workspace.default.bronze_chicago_inspections")
chicago_profile = profiler.profile(chicago)

dallas = spark.table("workspace.default.bronze_dallas_inspections")
dallas_profile = profiler.profile(dallas)

print("Both profiles generated!")

In [0]:
import pandas as pd

def parse_profile(profile_result):
    """Separate DQX profile output into stats dict and rules list"""
    stats = None
    rules = []
    for item in profile_result:
        if isinstance(item, dict):
            stats = item
        elif isinstance(item, list):
            rules.extend(item)
        else:
            rules.append(item)
    return stats, rules

def profile_stats_df(stats, exclude_prefix=None):
    """Convert stats dict to a displayable Spark DataFrame"""
    rows = []
    for col_name, col_stats in stats.items():
        if exclude_prefix and col_name.startswith(exclude_prefix):
            continue
        rows.append({
            "column": col_name,
            "count": col_stats.get("count", ""),
            "count_null": col_stats.get("count_null", ""),
            "count_non_null": col_stats.get("count_non_null", ""),
            "min": str(col_stats.get("min", "")),
            "max": str(col_stats.get("max", "")),
            "mean": str(col_stats.get("mean", "")),
            "stddev": str(col_stats.get("stddev", "")),
        })
    return spark.createDataFrame(pd.DataFrame(rows))

def profile_rules_df(rules, exclude_prefix=None):
    """Convert DQProfile rules to a displayable Spark DataFrame"""
    rows = []
    for r in rules:
        if not hasattr(r, "column"):
            continue
        if exclude_prefix and r.column.startswith(exclude_prefix):
            continue
        rows.append({
            "column": r.column,
            "rule": r.name,
            "description": r.description or "",
            "parameters": str(r.parameters) if r.parameters else ""
        })
    if not rows:
        return spark.createDataFrame(pd.DataFrame({"message": ["No rules extracted"]}))
    return spark.createDataFrame(pd.DataFrame(rows))

In [0]:
chicago_stats, chicago_rules = parse_profile(chicago_profile)

print("=== CHICAGO COLUMN STATISTICS ===")
display(profile_stats_df(chicago_stats))

print("=== CHICAGO DQX RULES ===")
display(profile_rules_df(chicago_rules))

In [0]:
dallas_stats, dallas_rules = parse_profile(dallas_profile)

print("=== DALLAS COLUMN STATISTICS (Core Columns) ===")
display(profile_stats_df(dallas_stats, exclude_prefix="violation_"))

print("=== DALLAS DQX RULES (Core Columns) ===")
display(profile_rules_df(dallas_rules, exclude_prefix="violation_"))

In [0]:
from pyspark.sql.functions import col, split, size, trim, length, explode, regexp_extract, when, expr, lit
from functools import reduce

def violation_summary(name, total, with_v):
    without_v = total - with_v
    print(f"\n=== {name} VIOLATIONS OVERVIEW ===")
    print(f"Total: {total} | With: {with_v} ({round(with_v/total*100,2)}%) | Without: {without_v} ({round(without_v/total*100,2)}%)")

# ============================================================================
# CHICAGO VIOLATIONS
# ----------------------------------------------------------------------------
# Chicago stores ALL violations for an inspection in a SINGLE TEXT COLUMN
# called "violations". Multiple violations are separated by pipe characters (|).
# Each violation follows the pattern: 
#   CODE_NUMBER. DESCRIPTION - Comments: INSPECTOR_NOTES
#
# To analyze individual violations, we must:
#   1. Split the text on " | " to get individual violation strings
#   2. Use regex to extract the code, description, and comments from each string
# ============================================================================

chicago = spark.table("workspace.default.bronze_chicago_inspections")
chi_total = chicago.count()
chi_with = chicago.where(col("violations").isNotNull() & (length(trim(col("violations"))) > 0)).count()
violation_summary("CHICAGO", chi_total, chi_with)

# Split the single pipe-delimited column into rows, then parse each violation
chi_parsed = (
    chicago.where(col("violations").isNotNull())
    .withColumn("violation_raw", explode(split(col("violations"), "\\|")))  # split pipe-delimited text into rows
    .withColumn("violation_raw", trim(col("violation_raw")))
    .withColumn("violation_code", regexp_extract("violation_raw", r"^(\d+)\.", 1))  # extract leading number
    .withColumn("violation_description", regexp_extract("violation_raw", r"^\d+\.\s*(.+?)\s*-\s*Comments:", 1))  # text before "- Comments:"
    .withColumn("violation_comments", regexp_extract("violation_raw", r"-\s*Comments:\s*(.*)", 1))  # text after "- Comments:"
)

# Count violations per inspection by counting pipe-separated segments
print("\n=== CHICAGO: VIOLATIONS PER INSPECTION ===")
display(chicago.where(col("violations").isNotNull()).withColumn("v_count", size(split(col("violations"), "\\|"))).groupBy("v_count").count().orderBy("v_count"))

print("\n=== CHICAGO: SAMPLE PARSED ===")
display(chi_parsed.select("inspection_id", "violation_code", "violation_description", "violation_comments").limit(20))

# ============================================================================
# DALLAS VIOLATIONS
# ----------------------------------------------------------------------------
# Dallas stores violations in a WIDE/DENORMALIZED FORMAT with 25 repeating
# column groups: violation_description_1..25, violation_points_1..25,
# violation_detail_1..25, violation_memo_1..25 (total: 100 violation columns).
#
# To analyze individual violations, we must:
#   1. UNPIVOT (stack) the 25 column groups into rows using SQL stack()
#   2. Drop rows where violation_description is null (empty slots)
#   3. Extract the violation code from the description prefix (e.g., "*31" -> 31)
#
# This is fundamentally different from Chicago because the data is already
# structured per-violation — we just need to reshape it from wide to long.
# Chicago requires TEXT PARSING; Dallas requires SCHEMA RESHAPING.
# ============================================================================

dallas = spark.table("workspace.default.bronze_dallas_inspections")
dal_total = dallas.count()

# Unpivot: convert 25 wide column groups into rows using stack()
# Each row becomes one violation with its description, points, detail, and memo
stack_expr = "stack(25, " + ", ".join([
    f"'{i}', violation_description_{i}, violation_points_{i}, violation_detail_{i}, violation_memo_{i}"
    for i in range(1, 26)
]) + ") as (slot, violation_description, violation_points, violation_detail, violation_memo)"

dal_unpivoted = (
    dallas.select("restaurant_name", "inspection_date", "inspection_score", expr(stack_expr))
    .where(col("violation_description").isNotNull())  # drop empty slots
    .withColumn("violation_code", regexp_extract("violation_description", r"\*?(\d+)", 1))  # extract code after optional asterisk
)

# Count violations per inspection using the unpivoted rows
dal_counts = dal_unpivoted.groupBy("restaurant_name", "inspection_date", "inspection_score").count().withColumnRenamed("count", "v_count")
dal_with = dal_counts.count()
violation_summary("DALLAS", dal_total, dal_with)

print("\n=== DALLAS: VIOLATIONS PER INSPECTION ===")
display(dal_counts.groupBy("v_count").count().orderBy("v_count"))

print("\n=== DALLAS: SAMPLE PARSED ===")
display(dal_unpivoted.select("restaurant_name", "inspection_date", "violation_code", "violation_description", "violation_points", "violation_memo").limit(20))

# ============================================================================
# ASSIGNMENT-SPECIFIC VALIDATION CHECKS (Dallas only)
# ----------------------------------------------------------------------------
# These checks only apply to Dallas per the assignment requirements:
#   1. Score >= 90 should not have more than 3 violations
#   2. Inspection result cannot be PASS if any violation contains Urgent/Critical
# 
# Chicago doesn't have a numeric score (it uses text results like "Pass"/"Fail"),
# so these score-based rules don't apply to Chicago. Chicago's score will be
# DERIVED later in the Silver layer based on the results-to-score mapping table.
# ============================================================================

# Check 1: Score >= 90 with more than 3 violations (reuses dal_counts from above)
print("\n=== DALLAS: SCORE >= 90 WITH > 3 VIOLATIONS (should be flagged) ===")
flagged = dal_counts.where((col("inspection_score") >= 90) & (col("v_count") > 3))
print(f"Count: {flagged.count()}")
display(flagged.limit(10))

# Check 2: Score >= 90 with Urgent/Critical violations (reuses dal_unpivoted from above)
print("\n=== DALLAS: SCORE >= 90 WITH URGENT/CRITICAL VIOLATIONS (should not be PASS) ===")
urgent = dal_unpivoted.where((col("inspection_score") >= 90) & col("violation_description").rlike("(?i)(urgent|critical)"))
print(f"Count: {urgent.count()}")
display(urgent.select("restaurant_name", "inspection_date", "inspection_score", "violation_description").limit(10))